In [3]:
import requests #Para fazer requisições HTTP e buscar dados da API
import tkinter as tk # Para criar a interface gráfica
from tkinter import ttk #Para usar widgets aprimorados do tkinter
from tkinter import messagebox #Para exibir mensagens de alerta e erro
from PIL import Image, ImageTk #Para trabalhar com imagens
import io # Para manipular dados binários, como imagens baixadas da internet

In [4]:

# Função para buscar o título em português
def buscar_titulo_em_portugues(filme_id, api_key):
    url = f"https://api.themoviedb.org/3/movie/{filme_id}?api_key={api_key}&language=pt-BR"
    response = requests.get(url)
    if response.status_code == 200:
        filme_detalhes = response.json()
        titulo_portugues = filme_detalhes.get("title", "")
        return titulo_portugues
    else:
        return ""

# Função para buscar detalhes do filme pelo ID em português
def buscar_detalhes_filme(filme_id, api_key):
    url = f"https://api.themoviedb.org/3/movie/{filme_id}?api_key={api_key}&language=pt-BR"
    response = requests.get(url)
    if response.status_code == 200:
        filme_detalhes = response.json()
        duracao = filme_detalhes.get("runtime", "")
        poster_path = filme_detalhes.get("poster_path", "")
        idioma_original = filme_detalhes.get("original_language", "")
        popularidade = filme_detalhes.get("popularity", "")
        paises_producao = ", ".join([country["name"] for country in filme_detalhes.get("production_countries", [])])
        sinopse = filme_detalhes.get("overview", "")
        nota_comunidade = filme_detalhes.get("vote_average", "")
        ano = filme_detalhes.get("release_date", "").split("-")[0] if filme_detalhes.get("release_date") else ""
        genero = ", ".join([genre["name"] for genre in filme_detalhes.get("genres", [])])
        return duracao, poster_path, idioma_original, popularidade, paises_producao, sinopse, nota_comunidade, ano, genero
    else:
        messagebox.showerror("Erro", f"Erro ao acessar a API TMDb: {response.status_code}")
        return "", "", "", "", "", "", "", "", ""

# Função para buscar dados do filme na API TMDb
def buscar_informacoes_filme(titulo):
    api_key = "915bc63d0621b761ee1453bbbb1f7ea0"  # Sua chave de API
    url = f"https://api.themoviedb.org/3/search/movie?api_key={api_key}&query={titulo}"
    
    response = requests.get(url)
    if response.status_code == 200:
        data = response.json()
        if data['results']:
            filme = data['results'][0]
            filme_id = filme.get("id", "")
            titulo_portugues = buscar_titulo_em_portugues(filme_id, api_key)  # Buscar título em português
            return filme_id, titulo_portugues
        else:
            messagebox.showwarning("Aviso", f"Nenhuma informação encontrada para o filme '{titulo}'")
            return "", ""
    else:
        messagebox.showerror("Erro", f"Erro ao acessar a API TMDb: {response.status_code}")
        return "", ""

# Função para exibir o poster do filme
def exibir_poster(poster_path, frame_poster):
    if poster_path:
        url_poster = f"https://image.tmdb.org/t/p/w500{poster_path}"
        response = requests.get(url_poster)
        if response.status_code == 200:
            img_data = response.content
            img = Image.open(io.BytesIO(img_data))
            img = img.resize((200, 300), Image.Resampling.LANCZOS)
            img_tk = ImageTk.PhotoImage(img)
            label_img = tk.Label(frame_poster, image=img_tk)
            label_img.image = img_tk  # Manter referência para evitar garbage collection
            label_img.pack()

# Função para mostrar detalhes do filme ao clicar na tabela
def mostrar_detalhes(event, tabela, detalhes_text, frame_poster, nota_var, assistido_var, local_entry, emocao_var, filmes_dados):
    selected_item = tabela.selection()[0]
    filme_id = tabela.item(selected_item, "values")[0]
    api_key = "915bc63d0621b761ee1453bbbb1f7ea0"  # Sua chave de API
    duracao, poster_path, idioma_original, popularidade, paises_producao, sinopse, nota_comunidade, ano, genero = buscar_detalhes_filme(filme_id, api_key)
    
    # Limpar texto e imagem anterior
    detalhes_text.delete(1.0, tk.END)
    for widget in frame_poster.winfo_children():
        widget.destroy()
    
    # Exibir os detalhes
    detalhes_text.insert(tk.END, f"Título Original: {filme_id}\n")
    detalhes_text.insert(tk.END, f"Ano: {ano}\n")
    detalhes_text.insert(tk.END, f"Gênero: {genero}\n")
    detalhes_text.insert(tk.END, f"Sinopse: {sinopse}\n\n")
    detalhes_text.insert(tk.END, f"Duração: {duracao} min\n")
    detalhes_text.insert(tk.END, f"Idioma Original: {idioma_original}\n")
    detalhes_text.insert(tk.END, f"Popularidade: {popularidade}\n")
    detalhes_text.insert(tk.END, f"Países de Produção: {paises_producao}\n")
    detalhes_text.insert(tk.END, f"Nota da Comunidade: {nota_comunidade}\n")
    
    exibir_poster(poster_path, frame_poster)

    # Carregar os campos de preenchimento manual
    if filme_id in filmes_dados:
        nota_var.set(filmes_dados[filme_id]['nota'])
        assistido_var.set(filmes_dados[filme_id]['assistido'])
        local_entry.delete(0, tk.END)
        local_entry.insert(0, filmes_dados[filme_id]['local'])
        emocao_var.set(filmes_dados[filme_id]['emocao'])
    else:
        nota_var.set(0)
        assistido_var.set("Não")
        local_entry.delete(0, tk.END)
        emocao_var.set("")

# Função para adicionar uma lista de filmes
def adicionar_lista_de_filmes(tabela, lista_filmes):
    filmes = lista_filmes.strip().split("\n")
    for filme in filmes:
        filme = filme.strip()  # Remove espaços em branco no início e no final
        if filme:  # Apenas processa se o filme não estiver vazio
            filme_id, titulo_portugues = buscar_informacoes_filme(filme)
            tabela.insert("", "end", values=(filme_id, titulo_portugues))

# Função para salvar os dados
def salvar_dados(tabela, nota_var, assistido_var, local_entry, emocao_var, filmes_dados):
    selected_item = tabela.selection()[0]
    filme_id = tabela.item(selected_item, "values")[0]
    nota = nota_var.get()
    assistido = assistido_var.get()
    local = local_entry.get()
    emocao = emocao_var.get()
    
    # Salvar os dados no dicionário
    filmes_dados[filme_id] = {
        'nota': nota,
        'assistido': assistido,
        'local': local,
        'emocao': emocao
    }
    print(f"Filme ID: {filme_id} - Dados salvos")

# Função para excluir um filme
def excluir_filme(tabela, filmes_dados):
    selected_item = tabela.selection()[0]
    filme_id = tabela.item(selected_item, "values")[0]
    tabela.delete(selected_item)
    if filme_id in filmes_dados:
        del filmes_dados[filme_id]

# Função para desenhar estrelas no canvas
def desenhar_estrelas(canvas, nota_var):
    canvas.delete("all")
    for i in range(10):
        x = i * 30 + 10
        y = 10
        if i < nota_var.get():
            fill_color = "yellow"
        else:
            fill_color = "gray"
        canvas.create_polygon([x, y+10, x+10, y+10, x+13, y, x+16, y+10, x+26, y+10, x+18, y+16, x+21, y+26, x+13, y+20, x+5, y+26, x+8, y+16],
                              fill=fill_color, outline="black")

def click_estrela(event, canvas, nota_var):
    x = event.x
    estrela_clicada = x // 30 + 1
    nota_var.set(estrela_clicada)
    desenhar_estrelas(canvas, nota_var)

# Função principal para criar a janela do aplicativo
def criar_janela_principal():
    janela = tk.Tk()
    janela.title("Mira")
    janela.geometry("1200x900")

    main_frame = tk.Frame(janela)
    main_frame.pack(fill="both", expand=True)

    canvas_scroll = tk.Canvas(main_frame)
    scrollbar = tk.Scrollbar(main_frame, orient="vertical", command=canvas_scroll.yview)
    scrollable_frame = tk.Frame(canvas_scroll)

    scrollable_frame.bind(
        "<Configure>",
        lambda e: canvas_scroll.configure(
            scrollregion=canvas_scroll.bbox("all")
        )
    )

    canvas_scroll.create_window((0, 0), window=scrollable_frame, anchor="nw")
    canvas_scroll.configure(yscrollcommand=scrollbar.set)

    canvas_scroll.pack(side="left", fill="both", expand=True)
    scrollbar.pack(side="right", fill="y")

    label = ttk.Label(scrollable_frame, text="Bem-vindo ao Mira!", font=("Helvetica", 16))
    label.pack(pady=20)

    colunas = ("ID", "Título em Português")
    tabela = ttk.Treeview(scrollable_frame, columns=colunas, show="headings")
    tabela.heading("ID", text="ID")
    tabela.heading("Título em Português", text="Título em Português")
    tabela.pack(pady=20, expand=True, fill="both")
    tabela.column("ID", width=0, stretch=tk.NO)  # Esconde a coluna de ID

    frame_adicionar = ttk.Frame(scrollable_frame)
    frame_adicionar.pack(pady=10)

    ttk.Label(frame_adicionar, text="Lista de Filmes:").grid(row=0, column=0, pady=5)
    lista_filmes_entry = tk.Text(frame_adicionar, height=5, width=50)
    lista_filmes_entry.grid(row=1, column=0, pady=5)

    adicionar_lista_btn = ttk.Button(frame_adicionar, text="Adicionar Lista de Filmes", 
                                     command=lambda: adicionar_lista_de_filmes(tabela, lista_filmes_entry.get("1.0", tk.END)))
    adicionar_lista_btn.grid(row=2, column=0, pady=10)

    # Botão para excluir um filme
    excluir_filme_btn = ttk.Button(frame_adicionar, text="Excluir Filme", command=lambda: excluir_filme(tabela, filmes_dados))
    excluir_filme_btn.grid(row=2, column=1, pady=10)

    # Área para exibir detalhes do filme
    detalhes_text = tk.Text(scrollable_frame, height=10, width=100)
    detalhes_text.pack(pady=10)

    frame_poster = tk.Frame(scrollable_frame)
    frame_poster.pack(pady=10)

    # Campos de preenchimento manual
    frame_manual = ttk.Frame(scrollable_frame)
    frame_manual.pack(pady=10, fill="x")

    nota_var = tk.IntVar()

    # Canvas para desenhar as estrelas
    estrelas_canvas = tk.Canvas(frame_manual, width=300, height=50)
    estrelas_canvas.grid(row=0, column=1, pady=5, sticky=tk.W)
    desenhar_estrelas(estrelas_canvas, nota_var)
    
    estrelas_canvas.bind("<Button-1>", lambda event: click_estrela(event, estrelas_canvas, nota_var))

    assistido_var = tk.StringVar()
    assistido_var.set("Não")
    ttk.Label(frame_manual, text="Assistido:").grid(row=1, column=0, pady=5, sticky=tk.W)
    assistido_check = ttk.Checkbutton(frame_manual, text="Sim", variable=assistido_var, onvalue="Sim", offvalue="Não")
    assistido_check.grid(row=1, column=1, pady=5, sticky=tk.W)

    ttk.Label(frame_manual, text="Onde foi assistido:").grid(row=2, column=0, pady=5, sticky=tk.W)
    local_entry = ttk.Entry(frame_manual, width=30)
    local_entry.grid(row=2, column=1, pady=5, sticky=tk.W)

    ttk.Label(frame_manual, text="Como me senti:").grid(row=3, column=0, pady=5, sticky=tk.W)
    emocao_var = tk.StringVar()
    emocao_combo = ttk.Combobox(frame_manual, textvariable=emocao_var)
    emocao_combo['values'] = ("😀 Feliz", "😢 Triste", "😠 Irritado", "😮 Surpreso", "😌 Relaxado")
    emocao_combo.grid(row=3, column=1, pady=5, sticky=tk.W)

    salvar_btn = ttk.Button(scrollable_frame, text="Salvar Dados", 
                            command=lambda: salvar_dados(tabela, nota_var, assistido_var, local_entry, emocao_var, filmes_dados))
    salvar_btn.pack(pady=10)

    # Bind para clicar no item da tabela
    tabela.bind("<ButtonRelease-1>", lambda event: mostrar_detalhes(event, tabela, detalhes_text, frame_poster, nota_var, assistido_var, local_entry, emocao_var, filmes_dados))

    # Dicionário para armazenar as informações dos filmes
    filmes_dados = {}

    janela.mainloop()

if __name__ == "__main__":
    criar_janela_principal()


Exception in Tkinter callback
Traceback (most recent call last):
  File "C:\Users\DSilva\anaconda3\Lib\tkinter\__init__.py", line 1968, in __call__
    return self.func(*args)
           ^^^^^^^^^^^^^^^^
  File "C:\Users\DSilva\AppData\Local\Temp\ipykernel_104940\618062820.py", line 250, in <lambda>
    tabela.bind("<ButtonRelease-1>", lambda event: mostrar_detalhes(event, tabela, detalhes_text, frame_poster, nota_var, assistido_var, local_entry, emocao_var, filmes_dados))
                                                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\DSilva\AppData\Local\Temp\ipykernel_104940\618062820.py", line 68, in mostrar_detalhes
    selected_item = tabela.selection()[0]
                    ~~~~~~~~~~~~~~~~~~^^^
IndexError: tuple index out of range
